# Notebook 02 — RBM Confidence Engine
**InterviewIQ · Units II–IV: latent variables, RBMs, Gibbs sampling, Contrastive Divergence.**

Visible units = 4 behaviour signals `[answer_score, fast_response, motion_calm, vocal_steadiness]`. The 2 hidden units are *latent variables* — never labelled, learned unsupervised. **Trained on synthetic profiles** (strong vs weak candidates); not validated against human ratings.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))   # so notebooks can import the project modules
DATA = '../data'
import numpy as np
from rbm import RBM, make_synthetic_behavior_data
from confidence import ConfidenceModel

## Train (Contrastive Divergence, CD-1, 300 epochs)

In [ ]:
np.random.seed(0)
data, labels = make_synthetic_behavior_data(n=600, seed=1)
rbm = RBM(n_visible=4, n_hidden=2, lr=0.05, seed=7)
rbm.train(data.copy(), epochs=300, k=1, batch_size=32, verbose=True)

## Did the hidden units discover the strong/weak structure by themselves?

In [ ]:
h = rbm.hidden_representation(data)
print('mean hidden activation, strong profiles:', h[labels==1].mean(axis=0).round(3))
print('mean hidden activation, weak profiles  :', h[labels==0].mean(axis=0).round(3))
print('\nlearned weights (rows = visible, cols = hidden):')
import pandas as pd
print(pd.DataFrame(rbm.W, index=['answer_score','fast_response','motion_calm','vocal_steadiness'], columns=['hidden0','hidden1']).round(2))

Each hidden unit specialises in one direction. So the confidence score is the **projection** of the hidden vector onto the strong↔weak line (not the average of the hidden units — that would be ≈0.5 for everyone).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'train_rbm.py'], cwd='..', check=True)   # saves weights + anchors for the app
cm = ConfidenceModel(f'{DATA}/rbm_weights.npz', f'{DATA}/rbm_anchors.npz')
for name, f in {'strong candidate':[0.9,1,0.85,0.9], 'average':[0.5,0.5,0.6,0.6], 'weak candidate':[0.2,0,0.2,0.3]}.items():
    print(f'{name:17s} -> confidence {cm.score(f)[0]:.0%}')

## Honest limits
- 'Confidence' has no objective ground truth; it is a latent variable inferred from 4 proxy signals.
- Training data is synthetic. Validation against human-rated data (MIT Interview Dataset / ChaLearn First Impressions) is future work.
- If camera/mic are off, those two inputs use the training-data mean (neutral).